In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
  import pandas as pd
  import numpy as np
  import seaborn as sns
  import matplotlib.pyplot as plt

In [3]:
src_minute = "/content/drive/MyDrive/smart_meter_dataset/household_power_consumption/MINUTE_power_consumption_NEW.csv"
src_minute

'/content/drive/MyDrive/smart_meter_dataset/household_power_consumption/MINUTE_power_consumption_NEW.csv'

In [4]:
df_minute = pd.read_csv(src_minute, infer_datetime_format=True, parse_dates=['Date_Time'])
df_minute.set_index('Date_Time', drop=False, inplace=True)
df_minute.head()

<ipython-input-4-fc35d7f59c41>:1: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_minute = pd.read_csv(src_minute, infer_datetime_format=True, parse_dates=['Date_Time'])


,Date_Time,Global_active_power,Global_reactive_power,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Global_active_power_MA30
Date_Time,,,,,,,,
2006-12-16 17:24:00,2006-12-16 17:24:00,4.216,0.418,18.4,0.0,1.0,17.0,NaN
2006-12-16 17:25:00,2006-12-16 17:25:00,5.360,0.436,23.0,0.0,1.0,16.0,NaN
2006-12-16 17:26:00,2006-12-16 17:26:00,5.374,0.498,23.0,0.0,2.0,17.0,NaN
2006-12-16 17:27:00,2006-12-16 17:27:00,5.388,0.502,23.0,0.0,1.0,17.0,NaN
2006-12-16 17:28:00,2006-12-16 17:28:00,3.666,0.528,15.8,0.0,1.0,17.0,NaN


In [5]:
df_minute[df_minute['Date_Time']=='2006-12-16 17:24:00']

,Date_Time,Global_active_power,Global_reactive_power,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Global_active_power_MA30
Date_Time,,,,,,,,
2006-12-16 17:24:00,2006-12-16 17:24:00,4.216,0.418,18.4,0.0,1.0,17.0,NaN


In [6]:
newdf=df_minute[['Global_active_power']]
newdf

,Global_active_power
Date_Time,
2006-12-16 17:24:00,4.216
2006-12-16 17:25:00,5.360
2006-12-16 17:26:00,5.374
2006-12-16 17:27:00,5.388
2006-12-16 17:28:00,3.666
...,...
2010-11-26 20:58:00,0.946
2010-11-26 20:59:00,0.944
2010-11-26 21:00:00,0.938


In [7]:
newdf.shape

(2075259, 1)

In [8]:
def series_to_supervised(data, n_in=1, n_out=1, dropnan=True):
    n_vars = 1 if type(data) is list else data.shape[1]
    dff = pd.DataFrame(data)
    cols, names = list(), list()
    for i in range(n_in, 0, -1):
        cols.append(dff.shift(-i))
        names += [('var%d(t-%d)' % (j+1, i)) for j in range(n_vars)]
    for i in range(0, n_out):
        cols.append(dff.shift(-i))
        if i==0:
            names += [('var%d(t)' % (j+1)) for j in range(n_vars)]
        else:
            names += [('var%d(t+%d)' % (j+1)) for j in range(n_vars)]
        agg = pd.concat(cols, axis=1)
        agg.columns = names
        if dropnan:
            agg.dropna(inplace=True)
        return agg

In [9]:
reframed = series_to_supervised(newdf.values, 90, 1)
reframed.head()

,var1(t-90),var1(t-89),var1(t-88),var1(t-87),var1(t-86),var1(t-85),var1(t-84),var1(t-83),var1(t-82),var1(t-81),...,var1(t-9),var1(t-8),var1(t-7),var1(t-6),var1(t-5),var1(t-4),var1(t-3),var1(t-2),var1(t-1),var1(t)
0,4.298,2.448,2.322,2.336,2.496,2.540,2.786,4.218,4.204,4.200,...,3.662,3.668,3.700,3.702,3.520,3.666,5.388,5.374,5.360,4.216
1,4.230,4.298,2.448,2.322,2.336,2.496,2.540,2.786,4.218,4.204,...,4.448,3.662,3.668,3.700,3.702,3.520,3.666,5.388,5.374,5.360
2,4.230,4.230,4.298,2.448,2.322,2.336,2.496,2.540,2.786,4.218,...,5.412,4.448,3.662,3.668,3.700,3.702,3.520,3.666,5.388,5.374
3,3.924,4.230,4.230,4.298,2.448,2.322,2.336,2.496,2.540,2.786,...,5.224,5.412,4.448,3.662,3.668,3.700,3.702,3.520,3.666,5.388
4,4.218,3.924,4.230,4.230,4.298,2.448,2.322,2.336,2.496,2.540,...,5.268,5.224,5.412,4.448,3.662,3.668,3.700,3.702,3.520,3.666


In [10]:
reframed.values.shape

(2075169, 91)

In [11]:
  df=reframed.iloc[0:1440]

In [12]:
train=reframed.drop("var1(t-90)",axis=1)
test=reframed[["var1(t-90)"]]
test

,var1(t-90)
0,4.298
1,4.230
2,4.230
3,3.924
4,4.218
...,...
2075164,0.946
2075165,0.944
2075166,0.938
2075167,0.934


In [33]:
x=train.iloc[0:1440]
y=test.iloc[0:1440]
y.columns[0]

'var1(t-90)'

In [14]:
!pip install river

  Using cached pandas-2.2.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (89 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 1.9 MB/s eta 0:00:00
Using cached pandas-2.2.3-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (13.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 22.3 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.11.4
    Uninstalling scipy-1.11.4:
      Successfully uninstalled scipy-1.11.4
  Attempting uninstall: pandas
    Found existing installation: pandas 2.1.4
    Uninstalling pandas-2.1.4:
      Successfully uninstalled pandas-2.1.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pycaret 3.3.2 requires pandas<2.2.0, but you have pandas 2.2.3 which is incompatible.
pycaret 3.3.2 requires scipy<=1.11.4,>=1.6.1, but you have scipy 1.15.3 which 

In [15]:
!pip install pycaret

  Using cached pandas-2.1.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (18 kB)
  Using cached scipy-1.11.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
Using cached pandas-2.1.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.2 MB)
Using cached scipy-1.11.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (36.4 MB)
  Attempting uninstall: scipy
    Found existing installation: scipy 1.15.3
    Uninstalling scipy-1.15.3:
      Successfully uninstalled scipy-1.15.3
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.3
    Uninstalling pandas-2.2.3:
      Successfully uninstalled pandas-2.2.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
river 0.22.0 requires pandas<3.0.0,>=2.2.3, but you have pandas 2.1.4 which is incompatible.
river 0.22.0 requires scipy<2.0

In [16]:
  import jinja2
  from pycaret.regression import setup, compare_models, pull, predict_model, finalize_model

Exception ignored on calling ctypes callback function: <function ThreadpoolController._find_libraries_with_dl_iterate_phdr.<locals>.match_library_callback at 0x7a972ff6d440>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/threadpoolctl.py", line 1005, in match_library_callback
    self._make_controller_from_path(filepath)
  File "/usr/local/lib/python3.11/dist-packages/threadpoolctl.py", line 1187, in _make_controller_from_path
    lib_controller = controller_class(
                     ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/threadpoolctl.py", line 114, in __init__
    self.dynlib = ctypes.CDLL(filepath, mode=_RTLD_NOLOAD)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/ctypes/__init__.py", line 376, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: dlopen() error


In [17]:
target = test.columns[0]
numeric_features = x.columns.tolist()
target

'var1(t-90)'

In [18]:
  from datetime import datetime

In [19]:
from pycaret.regression import setup, compare_models, finalize_model

# Setup the environment
s = setup(data = df, target = "var1(t-90)", session_id = 123)

# Use the 'include' parameter to limit the models being compared,
# or adjust 'fold' parameter to reduce computation. This won't exactly stop at 80%,
# but it's a way to manage computation time.
# Here, you can specify a subset of models to reduce the computation load.
# Adjusting this list or other parameters can indirectly affect the total computation time.
models_to_compare = ['et', 'xgboost', 'rf','gbr','huber','lr','ridge','br','lar','omp']   # Example: Only include linear regression, decision tree, and random forest

best = compare_models(include=models_to_compare, sort='MAE', fold=5)  # Adjust 'fold' to control execution time

final_best_model = finalize_model(best)


,Description,Value
0,Session id,123
1,Target,var1(t-90)
2,Target type,Regression
3,Original data shape,"(1440, 91)"
4,Transformed data shape,"(1440, 91)"
5,Transformed train set shape,"(1007, 91)"
6,Transformed test set shape,"(433, 91)"
7,Numeric features,90
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
et,Extra Trees Regressor,0.3514,0.3056,0.5516,0.7604,0.1982,0.3205,3.0200
rf,Random Forest Regressor,0.3765,0.3394,0.5817,0.7339,0.2084,0.3414,4.6980
xgboost,Extreme Gradient Boosting,0.3835,0.3656,0.6038,0.7129,0.2112,0.3305,2.3240
gbr,Gradient Boosting Regressor,0.4062,0.3780,0.6141,0.7032,0.2187,0.3606,1.9040
omp,Orthogonal Matching Pursuit,0.4317,0.4172,0.6443,0.6708,0.2386,0.3912,0.0520
huber,Huber Regressor,0.4417,0.4455,0.6664,0.6487,0.2470,0.3921,0.0840
br,Bayesian Ridge,0.4609,0.4495,0.6695,0.6464,0.2440,0.4108,0.0320
ridge,Ridge Regression,0.4681,0.4651,0.6809,0.6346,0.2459,0.4068,0.0300
lr,Linear Regression,0.4685,0.4660,0.6815,0.6339,0.2461,0.4069,0.0320
lar,Least Angle Regression,0.4862,0.4912,0.7001,0.6133,0.2506,0.4120,0.0660


Processing:   0%|          | 0/45 [00:00<?, ?it/s]

In [20]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_squared_log_error, mean_absolute_percentage_error

In [21]:
final_best_model.fit(x,y)

Pipeline(memory=Memory(location=None),
         steps=[('numerical_imputer',
                 TransformerWrapper(include=['var1(t-89)', 'var1(t-88)',
                                             'var1(t-87)', 'var1(t-86)',
                                             'var1(t-85)', 'var1(t-84)',
                                             'var1(t-83)', 'var1(t-82)',
                                             'var1(t-81)', 'var1(t-80)',
                                             'var1(t-79)', 'var1(t-78)',
                                             'var1(t-77)', 'var1(t-76)',
                                             'var1(t-75)', 'var1(t-74)',
                                             'var1(t-73)', 'var1(t-72)',
                                             'var1(t-71)', 'var1(t-7...
                                             'var1(t-65)', 'var1(t-64)',
                                             'var1(t-63)', 'var1(t-62)',
                                             'var1(t-61)', 'var1(t-60)', ...],
                                    transformer=SimpleImputer())),
                ('categorical_imputer',
                 TransformerWrapper(include=[],
                                    transformer=SimpleImputer(strategy='most_frequent'))),
                ('clean_column_names',
                 TransformerWrapper(transformer=CleanColumnNames())),
                ('actual_estimator',
                 ExtraTreesRegressor(n_jobs=-1, random_state=123))])

In [22]:
final_best_model.predict(x)

array([4.298, 4.23 , 4.23 , ..., 3.81 , 3.808, 3.978])

In [23]:
x=train.iloc[0:1440]
final_best_model.predict(x)

array([4.298, 4.23 , 4.23 , ..., 3.81 , 3.808, 3.978])

In [24]:
def automodel(data):
  s = setup(data = data, target = "var1(t-90)",session_id = 123)

  models_to_compare = ['et', 'xgboost', 'rf','gbr','huber','lr','ridge','br','lar','omp']
  best = compare_models(include=models_to_compare, sort='MAE', fold=5)
  #best = compare_models(sort='MAE')
  best_model = finalize_model(best)
  return best_model

In [25]:
from river.drift import PageHinkley, KSWIN, ADWIN

In [26]:
from river.drift import PageHinkley, ADWIN, KSWIN

# Method 1 - Page Hinkley
ph = PageHinkley(min_instances=60, threshold=5)

# Method 2 - Adaptive Window
ad = ADWIN(delta=0.004)

# Method 3 - Kolmogorov-Smirnov Windowing method
ks = KSWIN(alpha=0.001, window_size=500, stat_size=150, seed=None)

# Dictionary of drift detection methods
methods = {
    "page-hinkley": ph,
    "adaptive-window": ad,
    "Kolmogorov-Smirnov": ks
}

# Dictionary to track drift detection results
drift_det = {
    "page-hinkley": 0,
    "adaptive-window": 0,
    "Kolmogorov-Smirnov": 0
}

In [27]:
drift_count = 0  # Initialize drift counter outside the loops

for val in y.values.flatten():
    for i, dd in methods.items():
        dd.update(val)  # Add new value to the drift detector
        if dd.drift_detected:  # Check if drift is detected
            drift_count += 1  # Increment drift counter
            print(f"Drift Detected at {val}, Total Drifts: {drift_count}")

Drift Detected at 3.254, Total Drifts: 1
Drift Detected at 2.188, Total Drifts: 2
Drift Detected at 1.606, Total Drifts: 3
Drift Detected at 2.334, Total Drifts: 4
Drift Detected at 4.122, Total Drifts: 5
Drift Detected at 2.456, Total Drifts: 6
Drift Detected at 2.822, Total Drifts: 7
Drift Detected at 3.804, Total Drifts: 8
Drift Detected at 0.388, Total Drifts: 9
Drift Detected at 0.732, Total Drifts: 10
Drift Detected at 0.208, Total Drifts: 11
Drift Detected at 3.522, Total Drifts: 12
Drift Detected at 3.4, Total Drifts: 13
Drift Detected at 2.332, Total Drifts: 14
Drift Detected at 1.238, Total Drifts: 15
Drift Detected at 0.314, Total Drifts: 16
Drift Detected at 0.504, Total Drifts: 17
Drift Detected at 2.996, Total Drifts: 18
Drift Detected at 2.4, Total Drifts: 19
Drift Detected at 2.296, Total Drifts: 20
Drift Detected at 3.016, Total Drifts: 21
Drift Detected at 5.57, Total Drifts: 22
Drift Detected at 3.784, Total Drifts: 23
Drift Detected at 1.97, Total Drifts: 24
Drift D

In [29]:
from river import drift

In [32]:


k = 1441 * 2
d = 0

# Drift detector from River
adwin = drift.ADWIN()

for i in range(1, 2):  # One segment
    x = train.iloc[k * i:k * (i + 1)]
    y = test.iloc[k * i:k * (i + 1)]
    dfval = y.values.flatten()

    for j in range(1, 1440):
        val = dfval[j - 1]

        # Detect drift based on prediction error (if available) or actual value
        if j > 1:
            # Ensure input is a DataFrame with correct columns
            pred_val = final_best_model.predict(x.iloc[[j - 1]])[0]
            error = abs(val - pred_val)
            adwin.update(error)
        else:
            adwin.update(val)  # Use actual value for early initialization

        if adwin.drift_detected:
            h = train.iloc[k * i:(k * i) + j]
            hp = test.iloc[k * i:(k * i) + j]
            pred = final_best_model.predict(h)
            mae = mean_absolute_error(hp, pred)

            if mae > 0.2:
                print(f"⚠️ Drift detected and performance degraded below threshold (MAE = {mae:.3f}), so we are updating model based on AutoML")
                d = k * i + j
                windf = reframed.iloc[d - 1440:d]
                x_val = train.iloc[d - 1440:d]
                y_val = test.iloc[d - 1440:d]
                model = automodel(windf)
                model.fit(x_val, y_val)
                final_best_model = model
                adwin = drift.ADWIN()  # Reset ADWIN
                k = d
            else:
                print(f"✅ Drift detected but performance is good with MAE: {mae:.3f}")

⚠️ Drift detected and performance degraded below threshold (MAE = 0.342), so we are updating model based on AutoML


,Description,Value
0,Session id,123
1,Target,var1(t-90)
2,Target type,Regression
3,Original data shape,"(1440, 91)"
4,Transformed data shape,"(1440, 91)"
5,Transformed train set shape,"(1007, 91)"
6,Transformed test set shape,"(433, 91)"
7,Numeric features,90
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
et,Extra Trees Regressor,0.1058,0.0633,0.2499,0.9319,0.0896,0.0791,2.5340
huber,Huber Regressor,0.1097,0.0785,0.2774,0.9164,0.0956,0.0796,0.1400
rf,Random Forest Regressor,0.1107,0.0718,0.2636,0.9232,0.0932,0.0814,4.3040
gbr,Gradient Boosting Regressor,0.1209,0.0726,0.2660,0.9223,0.0941,0.0918,1.6860
xgboost,Extreme Gradient Boosting,0.1213,0.0788,0.2764,0.9154,0.0999,0.0920,1.4600
omp,Orthogonal Matching Pursuit,0.1306,0.0786,0.2777,0.9164,0.0973,0.1074,0.0320
br,Bayesian Ridge,0.1443,0.0821,0.2846,0.9127,0.1029,0.1225,0.0320
ridge,Ridge Regression,0.1480,0.0857,0.2910,0.9088,0.1049,0.1264,0.0320
lr,Linear Regression,0.1502,0.0872,0.2936,0.9072,0.1059,0.1288,0.0520
lar,Least Angle Regression,0.4064,0.5696,0.6691,0.3637,0.2134,0.4005,0.0420


Processing:   0%|          | 0/45 [00:00<?, ?it/s]

⚠️ Drift detected and performance degraded below threshold (MAE = 0.262), so we are updating model based on AutoML


,Description,Value
0,Session id,123
1,Target,var1(t-90)
2,Target type,Regression
3,Original data shape,"(1440, 91)"
4,Transformed data shape,"(1440, 91)"
5,Transformed train set shape,"(1007, 91)"
6,Transformed test set shape,"(433, 91)"
7,Numeric features,90
8,Preprocess,True
9,Imputation type,simple


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
et,Extra Trees Regressor,0.1391,0.1041,0.3170,0.9363,0.1010,0.1078,1.5560
rf,Random Forest Regressor,0.1424,0.1085,0.3229,0.9344,0.1018,0.1087,4.0840
xgboost,Extreme Gradient Boosting,0.1553,0.1236,0.3459,0.9253,0.1077,0.1235,1.3220
gbr,Gradient Boosting Regressor,0.1575,0.1143,0.3323,0.9307,0.1059,0.1317,1.5500
huber,Huber Regressor,0.1623,0.1362,0.3633,0.9175,0.1149,0.1361,0.0820
omp,Orthogonal Matching Pursuit,0.1795,0.1443,0.3728,0.9129,0.1202,0.1650,0.0560
br,Bayesian Ridge,0.1968,0.1518,0.3840,0.9081,0.1302,0.2073,0.0540
ridge,Ridge Regression,0.2104,0.1617,0.3970,0.9020,0.1367,0.2300,0.0360
lr,Linear Regression,0.2131,0.1638,0.3997,0.9007,0.1379,0.2344,0.0340
lar,Least Angle Regression,0.2717,0.2350,0.4702,0.8603,0.1683,0.3478,0.0680


Processing:   0%|          | 0/45 [00:00<?, ?it/s]

✅ Drift detected but performance is good with MAE: 0.168
